# Intro to Evidently

[Evidently](https://www.evidentlyai.com/) is an open-source Python package for evaluating, testing, and monitoring Machine Learning systems. In this notebook, we will compare a reference split against later data, inspect the resulting report, and save a reusable reference dataset for the batch monitoring chapter.


In [ ]:
import os
from pathlib import Path
from typing import Any, cast
from urllib.request import urlretrieve

import mlflow
import pandas as pd
from dotenv import load_dotenv
from evidently.future.datasets import DataDefinition, Dataset
from evidently.future.metrics import DriftedColumnsCount, MissingValueCount, ValueDrift
from evidently.future.report import Report
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

> **Note:** The first time you run the import cell above, you may see a `SyntaxWarning`. It comes from Evidently's own code, does not affect this notebook, and will not appear again.

Load your `.env` settings and connect to MLflow. This points the notebook at the tracking store from notebook 01, where your registered model lives.


In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get the MLflow tracking URI from environment variables
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")

# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("green-taxi-monitoring")

First we will load the model from the MLflow model registry. We reuse the `production` alias so the later notebooks and Prefect flow can all point to the same registered model name.


In [ ]:
def load_model(model_name):
    alias = "production"
    model_uri = f"models:/{model_name}@{alias}"
    model = mlflow.pyfunc.load_model(model_uri)
    return model

In [ ]:
model_name = "green-taxi-ride-duration"

loaded_model = load_model(model_name)

Then we prepare the data as in notebook 01:

In [ ]:
# Load the data
year = 2024
month = 1
color = "green"

df = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

In [ ]:
cat_features = ["PULocationID", "DOLocationID"]
num_features = ["trip_distance"]
target = "duration"

In [ ]:
# Apply the same duration, outlier, and category rules used in notebook 01
def preprocess(df):
    df["duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[df["trip_distance"].between(0.1, 100)]
    df[cat_features] = df[cat_features].astype(str)
    df = df[cat_features + num_features + [target]]
    return df

In [ ]:
df_processed = preprocess(df)

In [ ]:
y = df_processed["duration"]
X = df_processed.drop(columns=["duration"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

We will make the predictions on the train set first:

In [ ]:
y_pred_train = loaded_model.predict(X_train.to_dict(orient="records"))

train_data = X_train.copy()
train_data["duration"] = y_train
train_data["prediction"] = y_pred_train

Now on the test set:

In [ ]:
y_pred_test = loaded_model.predict(X_test.to_dict(orient="records"))

test_data = X_test.copy()
test_data["duration"] = y_test
test_data["prediction"] = y_pred_test

And calculate the mean absolute error (MAE):

In [ ]:
print(mean_absolute_error(train_data.duration, train_data.prediction))
print(mean_absolute_error(test_data.duration, test_data.prediction))

Finally, we will create the **Evidently report**. The first step is to prepare the data for Evidently:

In [ ]:
# Prepare Evidently datasets
val_data = test_data.copy()
data_definition = DataDefinition(
    numerical_columns=num_features + ["prediction"], categorical_columns=cat_features
)


train_dataset = Dataset.from_pandas(train_data, data_definition)

val_dataset = Dataset.from_pandas(val_data, data_definition)

Then we can create the report and include `ValueDrift`, `DriftedColumnsCount` and `MissingValueCount` to detect prediction drift and data drift. We set `include_tests=True` so the report also includes threshold-based checks.

In [ ]:
report_metrics: list[Any] = [
    cast(Any, ValueDrift)(column="prediction"),
    cast(Any, DriftedColumnsCount)(),
    cast(Any, MissingValueCount)(column="prediction"),
]

report = Report(
    metrics=report_metrics,
    include_tests=True,
)

Now we have to run the report with our data. We will use the train data as the reference and the test data as the current data:

In [ ]:
snapshot = report.run(reference_data=train_dataset, current_data=val_dataset)

In [ ]:
snapshot

We can also convert it to a dictionary and read the metrics:

In [ ]:
result = snapshot.dict()
result

In [ ]:
# Prediction drift
result["metrics"][0]["value"]

In [ ]:
# Number of drifted columns
result["metrics"][1]["value"]["count"]

In [ ]:
# Share of missing values
result["metrics"][2]["value"]["share"]

We do not see much drift between the train and test split. That is a good sign: the reference data and the holdout data still look similar enough that the report stays quiet.

But monitoring becomes more useful when we compare that reference against a later batch.


In [ ]:
# Load the data
year = 2025
month = 8
color = "green"

# Download the data
data_path = Path(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")
data_path.parent.mkdir(exist_ok=True)

if not data_path.exists():
    urlretrieve(
        f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet",
        data_path,
    )

df_2025_aug = pd.read_parquet(data_path)

Preprocess the data:

In [ ]:
df_processed_2025_aug = preprocess(df_2025_aug)

Predict on the new data:

In [ ]:
y_pred_2025_aug = loaded_model.predict(
    df_processed_2025_aug[cat_features + num_features].to_dict(orient="records")
)

In [ ]:
df_processed_2025_aug["prediction"] = y_pred_2025_aug

In [ ]:
# Prepare Evidently current dataset
current_dataset = Dataset.from_pandas(df_processed_2025_aug, data_definition)

And create the report again:

In [ ]:
snapshot = report.run(reference_data=train_dataset, current_data=current_dataset)

In [ ]:
snapshot

In [ ]:
result = snapshot.dict()
result

Here the report becomes more interesting: every monitored column has drifted, and the prediction drift score is well above the 0.1 threshold. That is a signal to investigate, not proof that prediction quality has degraded. When labels are available, compare MAE or RMSE on the current batch as well.

The scores for the numerical columns are smaller than the distances in notebook 02 because Evidently normalizes the Wasserstein distance by the spread of the reference data. That makes the scores unit-free, but it does not make one threshold appropriate for every column. We use Evidently's default 0.1 threshold here; in practice, calibrate thresholds against normal variation and the business context.


The final step is to save the training-side data with predictions as a reusable reference snapshot. The Prefect flow from chapter 04 will load this file and compare it to each new batch automatically.


In [ ]:
train_data.to_parquet("./data/reference_data.parquet")